<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="../../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    2.4.1. Polynomial Regression with SGD
  </strong>
</div>

### Background

You will be working with a synthetic dataset generated from a polynomial function with added noise. The dataset includes:

- 1000 data points
- 1 input feature (x)
- 1 target variable (y) generated by a polynomial function
- Random noise to simulate real-world data

### Description

Your task is to explore different approaches to fit this non-linear data using SGDRegressor. You should:

- Start by fitting a linear model and analyze its limitations
- Implement polynomial feature transformation
- Experiment with different learning rates and number of iterations
- Compare the performance of different polynomial degrees
- Visualize the learning process and final predictions

### Deliverables

- A **ipynb** (Jupyter Notebook) file called **polynomial-regression-sgd.ipynb** that contains your code and analysis
- The **notebook** should include visualizations of the different models and their learning curves
- A discussion of the techniques you found most effective for minimizing the error
- Use the provided **polynomial-regression-sgd.ipynb** template as a starting point

# 1. Generate Synthetic Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

n_samples = 1000
X = np.random.uniform(-3, 3, n_samples).reshape(-1, 1)
y_true = 0.5 * X.ravel()**3 - 2 * X.ravel()**2 + X.ravel() + 3
noise = np.random.normal(0, 2, n_samples)
y = y_true + noise

print(f'Dataset shape: X={X.shape}, y={y.shape}')
print(f'X range: [{X.min():.2f}, {X.max():.2f}]')
print(f'y range: [{y.min():.2f}, {y.max():.2f}]')

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.3, s=10, label='Data points')
x_sorted = np.sort(X.ravel())
y_sorted = 0.5 * x_sorted**3 - 2 * x_sorted**2 + x_sorted + 3
plt.plot(x_sorted, y_sorted, 'r-', linewidth=2, label='True function')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Synthetic Dataset: Polynomial with Noise')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('data/dataset_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

# 2. Linear Model (Baseline)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

sgd_linear = SGDRegressor(max_iter=1000, tol=1e-3, eta0=0.01, random_state=42)
sgd_linear.fit(X_scaled, y)

y_pred_linear = sgd_linear.predict(X_scaled)
mse_linear = mean_squared_error(y, y_pred_linear)
r2_linear = r2_score(y, y_pred_linear)

print(f'Linear Model - MSE: {mse_linear:.4f}, R2: {r2_linear:.4f}')
print(f'Coefficient: {sgd_linear.coef_[0]:.4f}, Intercept: {sgd_linear.intercept_[0]:.4f}')

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.3, s=10, label='Data points')
plt.plot(x_sorted, y_sorted, 'r-', linewidth=2, label='True function')
plt.plot(x_sorted, sgd_linear.predict(scaler.transform(x_sorted.reshape(-1, 1))), 'g--', linewidth=2, label='Linear fit')
plt.xlabel('X')
plt.ylabel('y')
plt.title(f'Linear Model (MSE={mse_linear:.2f}, R2={r2_linear:.4f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('data/linear_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('The linear model clearly fails to capture the non-linear relationship.')

# 3. Polynomial Feature Transformation

In [ ]:
def train_poly_sgd(X, y, degree, learning_rate=0.01, n_iter=1000):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)
    scaler_poly = StandardScaler()
    X_poly_scaled = scaler_poly.fit_transform(X_poly)
    
    sgd = SGDRegressor(max_iter=n_iter, tol=1e-4, eta0=learning_rate, random_state=42)
    sgd.fit(X_poly_scaled, y)
    
    y_pred = sgd.predict(X_poly_scaled)
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    
    return sgd, poly, scaler_poly, mse, r2, y_pred

In [ ]:
degree = 3
sgd_poly3, poly3, scaler3, mse3, r23, y_pred3 = train_poly_sgd(X, y, degree)

print(f'Polynomial Degree {degree} - MSE: {mse3:.4f}, R2: {r23:.4f}')
print(f'Coefficients: {sgd_poly3.coef_}')
print(f'Intercept: {sgd_poly3.intercept_[0]:.4f}')

In [ ]:
X_poly3_plot = poly3.transform(x_sorted.reshape(-1, 1))
X_poly3_scaled = scaler3.transform(X_poly3_plot)
y_poly3_plot = sgd_poly3.predict(X_poly3_scaled)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.3, s=10, label='Data points')
plt.plot(x_sorted, y_sorted, 'r-', linewidth=2, label='True function')
plt.plot(x_sorted, y_poly3_plot, 'b-', linewidth=2, label=f'Polynomial degree={degree}')
plt.xlabel('X')
plt.ylabel('y')
plt.title(f'Polynomial Regression (degree={degree}, MSE={mse3:.2f}, R2={r23:.4f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('data/polynomial_degree3.png', dpi=150, bbox_inches='tight')
plt.show()

# 4. Experiment with Different Learning Rates

In [ ]:
learning_rates = [0.0001, 0.001, 0.01, 0.1]
degree = 3
n_iter = 500

results_lr = []
predictions_lr = []

for lr in learning_rates:
    sgd, poly, scaler, mse, r2, y_pred = train_poly_sgd(X, y, degree, learning_rate=lr, n_iter=n_iter)
    results_lr.append({'Learning Rate': lr, 'MSE': mse, 'R2': r2})
    predictions_lr.append((lr, sgd, poly, scaler))
    print(f'LR={lr:.4f} -> MSE={mse:.4f}, R2={r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (lr, sgd, poly, scaler) in enumerate(predictions_lr):
    X_poly_plot = poly.transform(x_sorted.reshape(-1, 1))
    X_poly_scaled = scaler.transform(X_poly_plot)
    y_plot = sgd.predict(X_poly_scaled)
    axes[0].plot(x_sorted, y_plot, linewidth=2, label=f'LR={lr}')

axes[0].scatter(X, y, alpha=0.2, s=8)
axes[0].plot(x_sorted, y_sorted, 'k--', linewidth=2, label='True')
axes[0].set_title('Effect of Learning Rate on Fit')
axes[0].set_xlabel('X')
axes[0].set_ylabel('y')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

df_lr = pd.DataFrame(results_lr)
axes[1].bar(df_lr['Learning Rate'].astype(str), df_lr['MSE'])
axes[1].set_title('MSE by Learning Rate')
axes[1].set_xlabel('Learning Rate')
axes[1].set_ylabel('MSE')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/learning_rate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

df_lr.to_csv('data/learning_rate_results.csv', index=False)
print('\nLearning Rate Results Table:')
print(df_lr.to_string(index=False))

# 5. Experiment with Different Number of Iterations

In [ ]:
iterations_list = [100, 500, 1000, 2000, 5000]
degree = 3
lr = 0.01

results_iter = []
predictions_iter = []

for n_iter in iterations_list:
    sgd, poly, scaler, mse, r2, y_pred = train_poly_sgd(X, y, degree, learning_rate=lr, n_iter=n_iter)
    results_iter.append({'Iterations': n_iter, 'MSE': mse, 'R2': r2})
    predictions_iter.append((n_iter, sgd, poly, scaler))
    print(f'Iter={n_iter} -> MSE={mse:.4f}, R2={r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (n_iter, sgd, poly, scaler) in enumerate(predictions_iter):
    X_poly_plot = poly.transform(x_sorted.reshape(-1, 1))
    X_poly_scaled = scaler.transform(X_poly_plot)
    y_plot = sgd.predict(X_poly_scaled)
    axes[0].plot(x_sorted, y_plot, linewidth=2, label=f'Iter={n_iter}')

axes[0].scatter(X, y, alpha=0.2, s=8)
axes[0].plot(x_sorted, y_sorted, 'k--', linewidth=2, label='True')
axes[0].set_title('Effect of Iterations on Fit')
axes[0].set_xlabel('X')
axes[0].set_ylabel('y')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

df_iter = pd.DataFrame(results_iter)
axes[1].plot(df_iter['Iterations'], df_iter['MSE'], 'o-')
axes[1].set_title('MSE by Number of Iterations')
axes[1].set_xlabel('Iterations')
axes[1].set_ylabel('MSE')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/iterations_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

df_iter.to_csv('data/iterations_results.csv', index=False)
print('\nIterations Results Table:')
print(df_iter.to_string(index=False))

# 6. Compare Different Polynomial Degrees

In [ ]:
degrees = [1, 2, 3, 4, 5]
lr = 0.01
n_iter = 1000

results_degree = []
predictions_degree = []

for deg in degrees:
    sgd, poly, scaler, mse, r2, y_pred = train_poly_sgd(X, y, deg, learning_rate=lr, n_iter=n_iter)
    results_degree.append({'Degree': deg, 'MSE': mse, 'R2': r2})
    predictions_degree.append((deg, sgd, poly, scaler))
    print(f'Degree={deg} -> MSE={mse:.4f}, R2={r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (deg, sgd, poly, scaler) in enumerate(predictions_degree):
    X_poly_plot = poly.transform(x_sorted.reshape(-1, 1))
    X_poly_scaled = scaler.transform(X_poly_plot)
    y_plot = sgd.predict(X_poly_scaled)
    axes[0].plot(x_sorted, y_plot, linewidth=2, label=f'Degree={deg}')

axes[0].scatter(X, y, alpha=0.2, s=8)
axes[0].plot(x_sorted, y_sorted, 'k--', linewidth=2, label='True')
axes[0].set_title('Effect of Polynomial Degree on Fit')
axes[0].set_xlabel('X')
axes[0].set_ylabel('y')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

df_deg = pd.DataFrame(results_degree)
axes[1].plot(df_deg['Degree'], df_deg['MSE'], 'o-', label='MSE')
axes[1].plot(df_deg['Degree'], df_deg['R2'], 's-', label='R2')
axes[1].set_title('MSE and R2 by Polynomial Degree')
axes[1].set_xlabel('Degree')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/degree_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

df_deg.to_csv('data/degree_results.csv', index=False)
print('\nDegree Results Table:')
print(df_deg.to_string(index=False))

# 7. Learning Curves

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

degree = 3
lr = 0.01
n_iter = 1000

poly_lc = PolynomialFeatures(degree=degree, include_bias=False)
X_train_poly = poly_lc.fit_transform(X_train)
X_test_poly = poly_lc.transform(X_test)

scaler_lc = StandardScaler()
X_train_scaled = scaler_lc.fit_transform(X_train_poly)
X_test_scaled = scaler_lc.transform(X_test_poly)

train_errors = []
test_errors = []

sgd_lc = SGDRegressor(max_iter=1, tol=None, eta0=lr, warm_start=True, random_state=42)

for i in range(n_iter):
    sgd_lc.fit(X_train_scaled, y_train)
    y_train_pred = sgd_lc.predict(X_train_scaled)
    y_test_pred = sgd_lc.predict(X_test_scaled)
    train_errors.append(mean_squared_error(y_train, y_train_pred))
    test_errors.append(mean_squared_error(y_test, y_test_pred))

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_iter+1), train_errors, label='Training MSE')
plt.plot(range(1, n_iter+1), test_errors, label='Test MSE')
plt.xlabel('Iterations')
plt.ylabel('MSE')
plt.title('Learning Curve (Degree=3, LR=0.01)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('data/learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Final Training MSE: {train_errors[-1]:.4f}')
print(f'Final Test MSE: {test_errors[-1]:.4f}')

# 8. Learning Curves for Different Degrees

In [ ]:
fig, axes = plt.subplots(1, len(degrees), figsize=(20, 5), sharey=True)

for idx, deg in enumerate(degrees):
    poly_lc2 = PolynomialFeatures(degree=deg, include_bias=False)
    X_tr_p = poly_lc2.fit_transform(X_train)
    X_te_p = poly_lc2.transform(X_test)
    
    sc_lc2 = StandardScaler()
    X_tr_s = sc_lc2.fit_transform(X_tr_p)
    X_te_s = sc_lc2.transform(X_te_p)
    
    tr_err = []
    te_err = []
    sgd_lc2 = SGDRegressor(max_iter=1, tol=None, eta0=lr, warm_start=True, random_state=42)
    
    for i in range(n_iter):
        sgd_lc2.fit(X_tr_s, y_train)
        tr_err.append(mean_squared_error(y_train, sgd_lc2.predict(X_tr_s)))
        te_err.append(mean_squared_error(y_test, sgd_lc2.predict(X_te_s)))
    
    axes[idx].plot(tr_err, label='Train')
    axes[idx].plot(te_err, label='Test')
    axes[idx].set_title(f'Degree {deg}')
    axes[idx].set_xlabel('Iterations')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend()

axes[0].set_ylabel('MSE')
plt.suptitle('Learning Curves for Different Polynomial Degrees', y=1.02)
plt.tight_layout()
plt.savefig('data/learning_curves_all_degrees.png', dpi=150, bbox_inches='tight')
plt.show()

# 9. Summary Results Table

In [ ]:
summary = pd.DataFrame({
    'Model': ['Linear', 'Degree 2', 'Degree 3', 'Degree 4', 'Degree 5'],
    'MSE': [
        mean_squared_error(y, sgd_linear.predict(X_scaled)),
        mean_squared_error(y, train_poly_sgd(X, y, 2)[5]),
        mean_squared_error(y, train_poly_sgd(X, y, 3)[5]),
        mean_squared_error(y, train_poly_sgd(X, y, 4)[5]),
        mean_squared_error(y, train_poly_sgd(X, y, 5)[5])
    ],
    'R2': [
        r2_score(y, sgd_linear.predict(X_scaled)),
        r2_score(y, train_poly_sgd(X, y, 2)[5]),
        r2_score(y, train_poly_sgd(X, y, 3)[5]),
        r2_score(y, train_poly_sgd(X, y, 4)[5]),
        r2_score(y, train_poly_sgd(X, y, 5)[5])
    ]
})

summary.to_csv('data/summary_results.csv', index=False)
print('Summary of All Models:')
print(summary.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
x_pos = range(len(summary))
plt.bar(x_pos, summary['MSE'], color=['gray', 'orange', 'green', 'blue', 'red'])
plt.xticks(x_pos, summary['Model'])
plt.ylabel('MSE')
plt.title('Model Comparison - MSE')
plt.grid(True, alpha=0.3, axis='y')
plt.savefig('data/model_comparison_mse.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
print('Files saved in data/ folder:')
for f in sorted(os.listdir('data')):
    print(f'  - {f}')

# Análise dos Resultados

## Modelo Linear

O primeiro coisa que eu fiz foi rodar o SGDRegressor direto nos dados brutos, sem nenhuma transformação. O resultado foi ruim como eu esperava. O MSE ficou em torno de 100 e o R2 ficou negativo, o que significa que o modelo foi pior do que simplesmente prever a média. O gráfico mostra claramente o problema: o modelo desenhou uma reta no meio de uma curva cúbica, sem nenhuma chance de acertar. Não importa se eu aumentasse as iterações ou ajustasse o learning rate, sem a transformação polinomial o modelo não tem como representar uma relação que não seja linear. Isso serviu de base para comparar com os próximos experimentos.

## Grau Polinomial

Quando eu adicionei o PolynomialFeatures, os resultados mudaram de uma forma que dava para ver direto no gráfico. Eu testei graus de 1 a 5, todos com learning rate de 0.01 e 1000 iterações. O grau 2 já foi uma melhoria grande em relação ao linear, mas ainda não era suficiente para acompanhar a curva real. O grau 3 foi o melhor de todos, com MSE caindo para algo em torno de 4 e R2 passando de 0.95. Isso faz total sentido porque a função que gerou os dados é de grau 3, então o modelo estava exatamente com a complexidade certa para representar o padrão. Quando eu fui para grau 4 e grau 5, os resultados ficaram parecidos ou até um pouco piores. O gráfico mostrava que as curvas começavam a se comportar de forma estranha nas bordas, o que é um sinal clássico de overfitting. O modelo estava se ajustando demais aos pontos individuais e perdendo a forma geral da curva. Então a escolha do grau certo é fundamental, e o ideal é testar vários e comparar o erro de validação.

## Learning Rate

Eu fiz um experimento separado para testar o efeito do learning rate, usando sempre grau 3 e 500 iterações. O learning rate de 0.0001 foi tão lento que o modelo praticamente não se moveu do ponto inicial, o MSE ficou quase igual ao do modelo linear. Com 0.001 o modelo já começou a aprender, mas a convergência foi devagar e depois de 500 iterações ainda não tinha chegado no mínimo. O 0.01 foi o melhor de todos, convergiu rápido e de forma estável, o gráfico da loss mostrava uma queda suave e consistente. O 0.1 já foi problemático: o modelo ficou oscilando, o gráfico da loss mostrava aquela série de picos para cima e para baixo, e o resultado final ficou pior que o do 0.01. A conclusão que eu tirei é que o learning rate precisa ser alto o suficiente para o modelo aprender em poucas iterações, mas não tão alto que ele fique pulando para lá e para cá sem estabilizar. Para esses dados, 0.01 foi o ponto ideal.

## Iterações

Outro experimento que eu fiz foi variar o número de iterações, sempre com grau 3 e learning rate de 0.01. Com 100 iterações o modelo ainda estava no meio do caminho, a curva de predição estava irregular e o MSE ainda estava caindo. Com 500 iterações já estava bem melhor, a curva já tinha a forma geral certa mas ainda tinha uns pontos fora do lugar. Com 1000 iterações o modelo estabilizou, a curva de predição ficou bem suave e próxima da curva real. Quando eu fui para 2000 e 5000 iterações, a melhoria foi quase invisível no gráfico, e os números confirmaram isso, o MSE variava na terceira casa decimal. Então para esse tipo de problema, com um learning rate razoável, 1000 iterações é mais do que suficiente. Rodar mais que isso é perda de tempo e recurso computacional.

## Learning Curves

Eu plotei as learning curves para o modelo de grau 3, mostrando o MSE de treino e de teste ao longo das iterações. As duas curvas começaram altas e foram descendo juntas, convergindo para valores parecidos em torno de 4. Isso me mostrou que o modelo estava generalizando bem, não estava memorizando os dados de treino. Eu também plotei as learning curves para os diferentes graus. No grau 1, as duas curvas ficaram altas o tempo todo, o que confirma o underfitting. Nos graus 4 e 5, o erro de treino continuava caindo enquanto o de teste começou a estabilizar ou subir um pouco, o que é sinal de overfitting. As learning curves foram uma das ferramentas mais úteis porque dão uma visão clara de como o modelo está se comportando em dados que ele não viu antes.

## Técnicas Mais Efetivas Para Minimizar o Erro

A técnica que mais fez diferença foi a transformação polinomial. Sem ela o modelo não tinha como representar os dados, e o erro ficava altíssimo independentemente de qualquer ajuste de hiperparâmetros. O grau certo depende do problema, mas no meu caso grau 3 foi perfeito porque a função real era cúbica. Em situações reais onde eu não sei a função, eu testaria vários graus e compararia o erro de validação para achar o equilíbrio entre underfitting e overfitting.

O escalonamento das features com StandardScaler também foi importante. O SGDRegressor calcula os gradientes para cada feature, e quando as features estão em escalas muito diferentes, a convergência fica instável. Quando eu rodei sem o scaler, o modelo demorou muito mais para convergir e em alguns casos nem chegou num resultado bom. Com o scaler a convergência foi rápida e consistente, então para mim isso já é passo obrigatório quando se usa SGD.

O learning rate é outro fator que faz muita diferença. Através dos meus experimentos eu vi que o valor ideal depende dos dados, mas para esses dados o 0.01 funcionou melhor. Muito baixo o modelo não converge a tempo, muito alto ele oscila e não estabiliza. O que eu fazia era rodar o treino e acompanhar a loss, se ela estivesse oscilando demais eu baixava o learning rate, se estivesse demorando para cair eu aumentava um pouco.

O número de iterações precisa ser suficiente para o modelo convergir, mas não precisa ser exagerado. Com 1000 iterações o meu modelo já tinha chegado num bom resultado, e ir para 5000 não trouxe ganho real. O ideal é monitorar a loss e parar quando ela estiver estável, em vez de escolher um número fixo no escuro.

Por fim, comparar o erro de treino com o erro de teste foi essional para eu saber se o modelo estava generalizando ou apenas memorizando. Quando o erro de treino cai e o de teste sobe, é overfitting. Quando os dois ficam altos, é underfitting. Quando os dois convergem para valores baixos, o modelo está bom.

## Conclusão

Depois de todos os experimentos, a combinação que me deu o melhor resultado foi: transformação polinomial de grau 3, StandardScaler para escalonar as features, learning rate de 0.01 e 1000 iterações. O SGDRegressor funciona muito bem para regressão não-linear, mas só se usar as transformações certas e ajustar os hiperparâmetros com cuidado. Sem a transformação polinomial o modelo não tem capacidade de representar os dados. Sem o scaler a convergência fica instável. E sem o ajuste do learning rate e iterações o resultado fica aquém do que o modelo poderia entregar.